# 02 — QLoRA fine-tune of Qwen2.5-7B-Instruct

Prerequisite: `notebooks/01_explore_dataset.ipynb` has been run end-to-end and both gates passed. `data/train.parquet` and `data/val.parquet` exist, and `artifacts/gate_decisions.json` records the target model (7B default, 3B if yield was low).

Runtime: **Use GPU** (Runtime → Change runtime type → A100 or T4).
- A100 (Colab Pro): batch 2, grad-accum 4, ~20–30 min for 3 epochs.
- T4 (free): batch 1, grad-accum 8, ~1.5–2 h for 3 epochs.

In [ ]:
import os, sys

def _find_project_root() -> str:
    candidates = [os.path.abspath("..")]
    candidates += [
        "/content/fineTune LLM",
        "/content/fineTune_LLM",
        "/content/finetune-llm",
        "/content/drive/MyDrive/fineTune LLM",
        "/content/drive/MyDrive/fineTune_LLM",
    ]
    if os.path.isdir("/content"):
        for entry in os.listdir("/content"):
            p = os.path.join("/content", entry)
            if os.path.isfile(os.path.join(p, "schema.py")):
                candidates.append(p)
    for c in candidates:
        if os.path.isfile(os.path.join(c, "schema.py")):
            return c
    return ""

PROJECT_ROOT = _find_project_root()
if not PROJECT_ROOT:
    raise FileNotFoundError(
        "Project files not found. On Colab, either:\n"
        "  (A) Clone your repo:\n"
        "      !git clone https://github.com/<you>/<repo>.git '/content/fineTune LLM'\n"
        "  (B) Upload via the file panel (left sidebar) to /content/fineTune LLM/\n"
        "  (C) Mount Drive and copy from MyDrive.\n"
        "Then re-run this cell."
    )
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working in:", os.getcwd())


with open("artifacts/gate_decisions.json") as f:
    gate = json.load(f)
import json as _json
print(_json.dumps(gate, indent=2))

In [ ]:
# Unsloth install (CUDA-specific; handled here, not in requirements.txt)
%pip install -q --upgrade pip
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps "trl<0.14.0" "peft>=0.13.0" accelerate bitsandbytes
%pip install -q datasets pandas pyarrow pydantic jsonschema

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
is_a100 = torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0)

In [ ]:
from src.train import TrainConfig, train

# adapt batch size to GPU
cfg = TrainConfig(
    model_name=gate["target_model"],
    per_device_train_batch_size=2 if is_a100 else 1,
    gradient_accumulation_steps=4 if is_a100 else 8,
)
# name the output dir after the model so 7B and 3B runs don't clobber each other
slug = gate["target_model"].split("/")[-1].lower().replace(".", "")
cfg.output_dir = cfg.output_dir.parent / f"{slug}-meds-qlora"
print("Output dir:", cfg.output_dir)

train(cfg)

## Save to Drive (optional)

Colab runtimes disconnect; push the adapter to Drive so notebook 03 can pick it up without retraining.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    dest = f"/content/drive/MyDrive/fineTune_LLM/{cfg.output_dir.name}"
    os.makedirs(dest, exist_ok=True)
    for item in cfg.output_dir.iterdir():
        target = os.path.join(dest, item.name)
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)
    print("Copied adapter to:", dest)
except ImportError:
    print("Not on Colab — skipping Drive copy.")